In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ======================
# Load Data
# ======================
import pickle
import numpy as np

PKL_PATH = "/content/drive/MyDrive/colab_shared/outputs/DBF_reps.pkl"

with open(PKL_PATH, 'rb') as f:
    data = pickle.load(f)

print(f"Total reps: {len(data)}")

Total reps: 1692


In [ ]:
print(sum(d['label']==1 for d in data), sum(d['label']==0 for d in data))

933 759


In [ ]:

# ======================
# Prepare Data
# ======================
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

video_ids = [d['video_id'] for d in data]
frames = [d['frames'] for d in data]
labels = [d['label'] for d in data]

# 🔥 Split by video (correct approach)
unique_videos = list(set(video_ids))
train_videos, test_videos = train_test_split(
    unique_videos, test_size=0.2, random_state=42
)

X_train, y_train, X_test, y_test = [], [], [], []

for vid, frm, lbl in zip(video_ids, frames, labels):
    if vid in train_videos:
        X_train.append(frm)
        y_train.append(lbl)
    else:
        X_test.append(frm)
        y_test.append(lbl)


In [ ]:
# ======================
# Padding
# ======================
max_len = max(len(seq) for seq in frames)

X_train = pad_sequences(X_train, maxlen=max_len, dtype='float32', padding='post')
X_test  = pad_sequences(X_test,  maxlen=max_len, dtype='float32', padding='post')

y_train = np.array(y_train)
y_test  = np.array(y_test)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (1347, 297, 24)
Test shape: (345, 297, 24)


In [ ]:
# ======================
# Model
# ======================
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Masking(mask_value=0.0, input_shape=(max_len, X_train.shape[2])),

    tf.keras.layers.Conv1D(64, 3, activation='relu', padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(128, return_sequences=True)),
    tf.keras.layers.Dropout(0.4),

    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64)),
    tf.keras.layers.Dropout(0.4),

    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/masking.py:48: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'conv1d' (of type Conv1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ masking (Masking)               │ (None, 297, 24)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 297, 64)        │         4,672 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 297, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 297, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 297, 256)       │       197,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 297, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 392,257 (1.50 MB)

 Trainable params: 391,873 (1.49 MB)

 Non-trainable params: 384 (1.50 KB)

In [ ]:
# ======================
# Train
# ======================
history = model.fit(
    X_train, y_train,
    epochs=35,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/35
43/43 ━━━━━━━━━━━━━━━━━━━━ 64s 1s/step - accuracy: 0.7365 - loss: 0.5583 - precision: 0.7597 - recall: 0.7503 - val_accuracy: 0.7188 - val_loss: 0.6087 - val_precision: 0.8092 - val_recall: 0.6863
Epoch 2/35
43/43 ━━━━━━━━━━━━━━━━━━━━ 64s 2s/step - accuracy: 0.7431 - loss: 0.5453 - precision: 0.7641 - recall: 0.7599 - val_accuracy: 0.7594 - val_loss: 0.5850 - val_precision: 0.8040 - val_recall: 0.7843
Epoch 3/35
43/43 ━━━━━━━━━━━━━━━━━━━━ 84s 2s/step - accuracy: 0.7387 - loss: 0.5573 - precision: 0.7629 - recall: 0.7503 - val_accuracy: 0.7362 - val_loss: 0.5297 - val_precision: 0.8122 - val_recall: 0.7206
Epoch 4/35
43/43 ━━━━━━━━━━━━━━━━━━━━ 70s 2s/step - accuracy: 0.7535 - loss: 0.5290 - precision: 0.7693 - recall: 0.7778 - val_accuracy: 0.7043 - val_loss: 0.6397 - val_precision: 0.8188 - val_recall: 0.6422
Epoch 5/35
43/43 ━━━━━━━━━━━━━━━━━━━━ 65s 2s/step - accuracy: 0.7580 - loss: 0.5103 - precision: 0.7779 - recall: 0.7737 - val_accuracy: 0.7623 - val_loss: 0.5156 - val

In [ ]:
# ======================
# Evaluate
# ======================
loss, acc, pre, recall = model.evaluate(X_test, y_test, verbose=0)

print(f'✅ Test loss: {loss:.4f}')
print(f'✅ Accuracy: {acc:.4f}')
print(f'✅ Precision: {pre:.4f}')
print(f'✅ Recall: {recall:.4f}')

✅ Test loss: 0.4989
✅ Accuracy: 0.7391
✅ Precision: 0.8393
✅ Recall: 0.6912


In [ ]:
# ======================
# Save Model
# ======================
model.save("/content/drive/MyDrive/colab_shared/outputs/DBF_model_1_acc1_.keras")
print("✅ Model saved successfully")

✅ Model saved successfully


In [3]:
import pickle

with open("/content/drive/MyDrive/colab_shared/outputs/DBF_reps.pkl", 'rb') as f:
    data = pickle.load(f)

max_len = max(len(d['frames']) for d in data)
print("max_len =", max_len)

max_len = 297
